# Лабораторная работа 2

Методология ETL описывает цикл работы с данными для машинного обучения. В классическом бизнес-анализе (BI) ETL означает Extract, Transform, Load (Извлечение, Трансформация, Загрузка). Процесс может иметь свою специфику, особенно в части разметки данных.

Этап 1. EXTRACT — Сбор и извлечение данных.
На этом этапе данные собираются из различных источников (часто разрозненных) и сводятся в единое сырое хранилище.

Этап 2. TRANSFORM — Очистка, Подготовка и Разметка.
Это самый сложный и важный этап, так как модели учатся только на тех данных, которые мы им дадим (принцип "Garbage In, Garbage Out")

    В ML-контексте этот этап делится на три подэтапа:
    1. Очистка от дубликатов, шума, выборосов (Data Cleaning).
    2. Подготовка признаков и генерация новых (Feature Engineering & Preprocessing), 
    3. Подготовка целевого признака, разметка (Labeling / Annotation), которую модель должна предсказывать.

Этап 3. LOAD — Загрузка в хранилище.
Готовые, очищенные и размеченные данные нужно доставить туда, откуда они будут попадать в модель.

Стек:
- pandas
- sqlalchemy

- 'A' - создать ячейку выше, 'B' - создать ячейку ниже
- 'Z' - отменить удаление ячейки
- Shift + Enter - выполнить ячейку
- Изменить тип ячеки - заменить 'Code' на 'Markdown'

# Подготовка

## Подготовка проекта и окружения

При первом запуске установите необходимые библиотеки.
Раскомментируйте строку и запустите ячейку

In [ ]:
# !uv add pandas pyarrow sqlalchemy scikit-learn seaborn

## Подготовка БД

Для наглядности этапа 1 по сбору и извлечении данных создадим простейшую базу данных на основе готового файла.

Запустите ячейки ниже ПЕРЕД НАЧАЛОМ РАБОТЫ.

In [ ]:
import os 
import pandas as pd
from sqlalchemy import create_engine, inspect

In [ ]:
# Путь к файлу
RAW_DATA_PATH = 'data/data_lab02.parquet'
# Название таблицы БД
DB_NAME = 'data_lab02'
# Путь к файлу БД
DB_PATH = f'data/db/{DB_NAME}.db'
# Создание папки, если не существует
os.makedirs('data/db', exist_ok=True)

# 1. Читаем файл
df_raw = pd.read_parquet(RAW_DATA_PATH)

# 2. Создаём SQLAlchemy engine для SQLite
engine = create_engine(f"sqlite:///{DB_PATH}")

# 3. Записываем данные в таблицу DB_NAME
with engine.begin() as conn:
    df_raw.to_sql(
        name=DB_NAME,
        con=conn,
        if_exists="replace",  # если таблица была — удаляем и создаём заново
        index=False
    )

print(f"Таблица '{DB_NAME}' создана в базе {DB_PATH}")

# 4. Проверяем структуру таблицы через SQLAlchemy Inspector
inspector = inspect(engine)
columns = inspector.get_columns(DB_NAME)

print("\n Структура таблицы (SQLAlchemy):")
for col in columns:
    print(f"   - {col['name']} ({col['type']})")

# Закрываем пул соединений
engine.dispose()


# Самостоятельная работа

У вас есть два источника данных:
1) файл со словарём станций
2) история перевозок

Задание:
- загрузите и откройте оба источника
- изучите и подготовьте данные (преобразуйте типы, объедините в одну таблицу)
- сохраните результат в файл и в БД

## Этап 1. Сбор и извлечение данных

### 1.1. Изучите данные из первого источника

Скачайте из базы данных. Напишите недостающий код и запустите ячейку

In [ ]:
# Простая загрузка всей таблицы
# engine = # YOUR CODE

query_advanced = f"""
    SELECT *
    FROM {DB_NAME}
"""
# df = # YOUR CODE

Изучите данные.

Используйте документацию к pandas. 
Ответы предоставьте в ячейках после вопроса 
(в виде кода или текстовых комментариев).

`Какой размер таблицы? Сколько признаков (столбцов) и примеров (строк)?`

`Какие типы данных у столбцов? Сколько столбцов каждого типа?`

_Подсказка_  Используйте документацию к pandas для функций: head, shape, info, describe 

### 1.2. Изучите данные из второго источника

Откройте файл, в котором словарь станций. Запустите ячейку

In [ ]:
df_stations = pd.read_parquet('data/station_reference.parquet')

Изучите данные.

Используйте код, чтобы ответить на вопросы ниже. 
Ответы предоставьте в ячейках после вопроса

`Какой размер таблицы? Сколько признаков (столбцов) и примеров (строк)?`

`Какие типы данных у столбцов? Сколько столбцов каждого типа?`

_Подсказка_  Используйте документацию к pandas для функций: head, shape, info, describe

### 1.3. Изучите пропуски в данных

Наличие пустых значений (None, pd.NA и т.п. ) в таблицах может повлиять на объединение двух источников данных:
- Коварное свойство: NaN не равен NaN
- Может приводить к потерям или дублям в итоговой таблице
- Если в колонке-ключе появляется pd.NA, pandas часто автоматически меняет тип колонки с int на float (чтобы вместить NaN) или на object

`Посчитайте в df доли пропусков`

`Посчитайте в df_stations доли пропусков`

`Удалите строки, для которых 'Код станции отправления' или 'Код станции назначения' пустой`

`Удалите столбцы, в которых есть пустые значениями`

_Подсказка_: 
- Изучите в документации pandas функцию notna и isna. 
- К isna дополнительно можно применить функции any, sum, mean.
- Список столбцов можно получить через columns
- Обращение к данным возможно через df[mask], где mask - список булевых величин длины df

### Объединение данных

В данных есть код станции отправления и назначения, но нет названий. 
Добавьте их из словаря.

`Проверьте типы столбцов с кодом в обоих источниках.  
Приведите к строковому виду с 6 знаками 
(если цифр меньше, заполняйте нулями слева)`

_Подсказка_: 
- int преобразует в целое, отсекая дробную часть
- в f-строке внутри фигурных скобок {...} можно указать формат после двоеточия ':' в виде {выражение:спецификатор_формата}.  
Пример: f"{42:0>5}"



`Объедините данные со словарём по коду отправления. Добавьте название станции. Переименуйте столбец в 'Станция отправления'.`

_Совет_: Используйте пример ниже. Убедитесь, что объединение работает правильно и количество строк в данных не изменилось. 

In [ ]:
# Пример объединения
print(f"Размер до:    {df.shape}")

df_temp = pd.merge(
    df,
    df_stations[['Код станции', 'Станция']],
    how='left',
    left_on='Код станции отправления',
    right_on='Код станции',
)
df_temp = df_temp.drop(columns=['Код станции'])
df_temp = df_temp.rename(columns={'Станция': 'Станция отправления'})

print(f"Размер после: {df_temp.shape}")

In [ ]:
df = df_temp.copy()

`Объедините данные со словарём по коду назначения. Добавьте название станции. Переименуйте столбец в 'Станция назначения'`

### Фильтрация

`Оставьте только те строки, где Сумма >= 100`  
`Проверьте размер таблицы до и после фильтрации`

`ДФЭ - двадцатифутовый эквивалент (основанная на стандартном 20-футовом морском контейнере).`  
`Оставьте только те строки, где ДФЭ > 0`  
`Проверьте размер таблицы до и после фильтрации`

_Подсказка_: 
- Обращение к данным возможно через df[mask], где mask - список булевых величин длины df
- Пример: mask = df['столбец_1'] != 0

## Выгрузка в БД

`Сохраните результат в базу данных. Выполните ячейки ниже`

In [ ]:
# Название таблицы БД
DB_NAME_PREPARED = "data_lab02_prepared"
# Путь к файлу БД
DB_PATH_PREPARED = f'data/db/{DB_NAME_PREPARED}.db'

In [ ]:
# Записываем данные в таблицу DB_NAME_PREPARED
engine = create_engine(f"sqlite:///{DB_PATH_PREPARED}")
with engine.begin() as conn:
    df.to_sql(
        name=DB_NAME_PREPARED,
        con=conn,
        if_exists="replace",  # если таблица была — удаляем и создаём заново
        index=False
    )

print(f"Таблица '{DB_NAME_PREPARED}' создана в базе {DB_PATH_PREPARED}")

In [ ]:
# Проверяем структуру таблицы через SQLAlchemy Inspector
inspector = inspect(engine)
columns = inspector.get_columns(DB_NAME_PREPARED)

print("\n Структура таблицы (SQLAlchemy):")
for col in columns:
    print(f"   - {col['name']} ({col['type']})")

# Закрываем пул соединений
engine.dispose()

`Дополнительно сохраните результат в файл parquet (функция to_parquet)`